# Prediksjon av liggetid ved akuttinnleggelse – Random Forest

## Formål

Formålet med analysen er å undersøke om lengden på sykehusopphold ved akuttinnleggelse kan predikeres ved hjelp av seks laboratorieprøver tatt nær innleggelsestidspunktet, samt pasientens alder.

Analysen er gjennomført som del av studentprosjektet i emnet IT6117 – Health Data Analyses with AI ved NTNU.

## Datagrunnlag

Dataene er hentet fra MIMIC-IV (versjon 3.1), modulen `hosp`, og bearbeidet i Google BigQuery.

Det endelige analysedatasettet omfatter 56 917 akuttinnleggelser med komplette verdier for seks laboratorieprøver: kreatinin, glukose, hemoglobin, kalium, natrium og leukocytter.

Laboratorieprøvene er valgt innenfor et tidsvindu på seks timer før til seks timer etter innleggelse.

## Metode

Random Forest-regresjon benyttes som hovedmetode for å predikere liggetid (LOS) i dager. Analysen gjennomføres i Python ved hjelp av Google Colab og scikit-learn.

Modellens prediksjonsevne evalueres ved hjelp av Mean Absolute Error (MAE) og sammenlignes med en enkel referansemodell basert på median liggetid.


## 1. Tilkobling og innlasting av analysedata

Kjør cellene nedenfor i Colab for å autentisere deg og hente analysedatasettet fra BigQuery.


In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:

from google.cloud import bigquery

client = bigquery.Client(project="arnhildmimic")

query = """
SELECT *
FROM `arnhildmimic.mimic_analyse.analyse_data`
"""

df = client.query(query).to_dataframe()

print("Antall rader og kolonner:", df.shape)
df.head()


Antall rader og kolonner: (56917, 10)


,subject_id,hadm_id,alder,kreatinin,glukose,hemoglobin,kalium,natrium,leukocytter,los
0,10296472,20967375,43,6.5,95.0,8.7,1.5,130.0,7.0,7.786111
1,12224263,23294937,52,0.1,75.0,6.6,1.7,148.0,5.3,1.865278
2,16573207,21399919,42,0.5,919.0,7.4,1.8,176.0,7.6,2.717361
3,10547408,27351978,43,0.4,114.0,9.9,1.8,121.0,13.6,15.970833
4,16878717,20838215,78,0.2,100.0,11.0,1.8,147.0,5.7,5.694444


## 2. Variabler og første oppdeling i trenings- og testdata

De syv prediktorene er alder og seks laboratorieprøver. `GroupShuffleSplit` deler etter `subject_id` for å holde innleggelser fra samme pasient atskilt.


In [ ]:

from sklearn.model_selection import GroupShuffleSplit

variabler = [
    "alder", "kreatinin", "glukose",
    "hemoglobin", "kalium", "natrium",
    "leukocytter"
]

X = df[variabler].astype(float)
y = df["los"].astype(float)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=df["subject_id"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Treningsdata:", len(X_train))
print("Testdata:", len(X_test))
print("Antall variabler:", X_train.shape[1])


Treningsdata: 45554
Testdata: 11363
Antall variabler: 7


## 3. Innledende Random Forest-modell

Den første modellen trenes på treningsutvalget og vurderes på testutvalget. Dette er en del av den innledende utforskningen.


In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Opprett Random Forest-modellen
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Tren modellen
rf.fit(X_train, y_train)

# Prediker liggetid for testdataene
y_pred = rf.predict(X_test)

# Beregn gjennomsnittlig prediksjonsfeil
mae = mean_absolute_error(y_test, y_pred)

print("Random Forest er ferdig trent!")
print("MAE:", round(mae, 2), "dager")


Random Forest er ferdig trent!
MAE: 5.49 dager


## 4. Første sammenligning med referansemodell

Referansemodellen predikerer median liggetid fra treningsutvalget for alle innleggelser. Begge modeller vurderes med MAE på samme testutvalg.


In [ ]:

import numpy as np
from sklearn.metrics import mean_absolute_error

# Referansemodell: Prediker median liggetid
median_los = y_train.median()

# Samme prediksjon for alle i testdatasettet
baseline_pred = np.full(len(y_test), median_los)

# Beregn prediksjonsfeil
baseline_mae = mean_absolute_error(
    y_test, baseline_pred
)

print("Median liggetid i treningsdata:",
      round(median_los, 2), "dager")

print("MAE referansemodell:",
      round(baseline_mae, 2), "dager")

print("MAE Random Forest:",
      round(mae, 2), "dager")


Median liggetid i treningsdata: 4.61 dager
MAE referansemodell: 4.74 dager
MAE Random Forest: 5.49 dager


## 5. Undersøkelse av overtilpasning og enklere modell

Vi sammenligner feil på trenings- og testdata og prøver en modell med begrenset kompleksitet.


In [ ]:

# Prediksjoner på treningsdataene
y_pred_train = rf.predict(X_train)

# Beregn treningsfeil
mae_train = mean_absolute_error(
    y_train, y_pred_train
)

print("MAE treningsdata:",
      round(mae_train, 2), "dager")

print("MAE testdata:",
      round(mae, 2), "dager")


MAE treningsdata: 2.0 dager
MAE testdata: 5.49 dager


In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Random Forest med begrenset kompleksitet
rf_enkel = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

# Tren modellen
rf_enkel.fit(X_train, y_train)

# Prediksjoner
pred_train = rf_enkel.predict(X_train)
pred_test = rf_enkel.predict(X_test)

# Beregn feil
mae_train_enkel = mean_absolute_error(
    y_train, pred_train
)
mae_test_enkel = mean_absolute_error(
    y_test, pred_test
)

print("MAE treningsdata:",
      round(mae_train_enkel, 2), "dager")
print("MAE testdata:",
      round(mae_test_enkel, 2), "dager")


MAE treningsdata: 4.94 dager
MAE testdata: 5.12 dager


## 6. Eget valideringsutvalg

Den opprinnelige treningsdelen deles videre i nytt treningsutvalg og valideringsutvalg, fortsatt gruppert etter pasient. De registrerte størrelsene var 36 493 trening, 9 061 validering og 11 363 test.

**Metodisk begrensning:** Testutvalget ble allerede brukt under innledende modellutvikling. Det er derfor ikke et helt uberørt slutt-testutvalg, og resultater fra dette må tolkes med forsiktighet.


In [ ]:

from sklearn.model_selection import GroupShuffleSplit

# Del de opprinnelige treningsdataene
splitter_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

# Pasient-ID for treningsdataene
pasienter_train = df.iloc[train_idx]["subject_id"]

idx_train, idx_val = next(
    splitter_val.split(
        X_train,
        y_train,
        groups=pasienter_train
    )
)

# Nye trenings- og valideringsutvalg
X_train_ny = X_train.iloc[idx_train]
X_val = X_train.iloc[idx_val]

y_train_ny = y_train.iloc[idx_train]
y_val = y_train.iloc[idx_val]

print("Treningsutvalg:", len(X_train_ny))
print("Valideringsutvalg:", len(X_val))
print("Testutvalg:", len(X_test))


Treningsutvalg: 36493
Valideringsutvalg: 9061
Testutvalg: 11363


## 7. Sammenligning av ulike tredybder på valideringsdata

Tre Random Forest-modeller med maksimal dybde 3, 5 og 10 sammenlignes ved MAE på valideringsutvalget.


In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

resultater = []

for dybde in [3, 5, 10]:
    modell = RandomForestRegressor(
        n_estimators=100,
        max_depth=dybde,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    )

    modell.fit(X_train_ny, y_train_ny)

    prediksjon = modell.predict(X_val)
    feil = mean_absolute_error(y_val, prediksjon)

    resultater.append((dybde, feil))

    print(
        "Maks dybde:", dybde,
        "- MAE:", round(feil, 2), "dager"
    )


Maks dybde: 3 - MAE: 5.07 dager
Maks dybde: 5 - MAE: 5.02 dager
Maks dybde: 10 - MAE: 4.99 dager


## 8. Referansemodell mot beste Random Forest på valideringsdata

Referansemodellen bruker median liggetid fra det **nye treningsutvalget**, og sammenlignes med laveste validerings-MAE blant de tre Random Forest-modellene.

**Registrerte resultater:** Referansemodell 4,63 dager; Random Forest 4,99 dager (maks dybde 10). Lavere MAE er bedre. Referansemodellen gjorde det dermed bedre på dette valideringsutvalget.


In [ ]:

import numpy as np
from sklearn.metrics import mean_absolute_error

# Median liggetid i det nye treningsutvalget
median_train = y_train_ny.median()

# Referansemodellen predikerer samme verdi for alle
pred_baseline_val = np.full(len(y_val), median_train)

# Beregn MAE på valideringsutvalget
mae_baseline_val = mean_absolute_error(
    y_val, pred_baseline_val
)

print("Median liggetid:", round(median_train, 2))
print("MAE referansemodell:",
      round(mae_baseline_val, 2), "dager")
print("MAE Random Forest:",
      round(min(feil for dybde, feil in resultater), 2),
      "dager")


Median liggetid: 4.63
MAE referansemodell: 4.63 dager
MAE Random Forest: 4.99 dager


## 9. Variabelbetydning i Random Forest

Modellen med maksimal dybde 10 brukes til å undersøke hvilke variabler som bidrar mest til oppdelingene i trærne. Variabelbetydning er ikke det samme som årsakssammenheng.


In [ ]:

import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Tren modellen med valgt tredybde
rf_valgt = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

rf_valgt.fit(X_train_ny, y_train_ny)

# Hent variablenes betydning
betydning = pd.DataFrame({
    "Variabel": X_train_ny.columns,
    "Betydning": rf_valgt.feature_importances_
})

betydning = betydning.sort_values(
    "Betydning", ascending=False
)

print(betydning)


      Variabel  Betydning
6  leukocytter   0.275647
3   hemoglobin   0.192943
0        alder   0.126034
2      glukose   0.118849
5      natrium   0.097954
1    kreatinin   0.096991
4       kalium   0.091583


## 10. Foreløpig tolkning og begrensninger

Random Forest ga ikke lavere MAE enn den enkle medianbaserte referansemodellen i valideringsutvalget. Dette tyder på at de valgte prediktorene og modellinnstillingene ikke ga en forbedring i denne evalueringen.

Resultatet bør diskuteres sammen med mulige skjevheter ved å inkludere bare innleggelser med alle seks laboratorieprøver, ekstreme LOS-verdier og at testutvalget ble brukt i innledende modellutvikling. Det er ikke gjennomført noen ny uavhengig sluttevaluering i denne notatboken.
